In [2]:
vocab = ["king", "queen", "man", "woman", "apple"]
word_to_index = {word:index for index, word in enumerate(vocab)}
word_to_index

{'king': 0, 'queen': 1, 'man': 2, 'woman': 3, 'apple': 4}

# One Hot Encoding

In [ ]:
import torch

def one_hot_encoding(word, vocab_size):
    vector = torch.zeros(vocab_size)
    vector[word_to_index[word]] = 1
    return vector

print(f"Vector of king : {one_hot_encoding('king', len(vocab))}")
print(f"Vector of queen: {one_hot_encoding('queen', len(vocab))}")



Vector of king : tensor([1., 0., 0., 0., 0.])
Vector of queen: tensor([0., 1., 0., 0., 0.])


# Manually designed embeddings

In [9]:
manual_embeddings = {
    "king": torch.tensor([1.0, 1.0, 1.0, 0.0]),
    "queen": torch.tensor([1.0, -1.0, 1.0, 0.0]),
    "man": torch.tensor([0.0, 1.0, 1.0, 0.0]),
    "woman": torch.tensor([0.0, -1.0, 1.0, 0.0]),
    "apple": torch.tensor([0.0, 0.0, 0.0, 1.0]),
}

## Similarity Using Dot Product

In [14]:
def similarity(w1, w2):
    return torch.dot(manual_embeddings[w1], manual_embeddings[w2])

print("king vs queen:", similarity("king", "queen"))
print("king vs apple:", similarity("king", "apple"))

king vs queen: tensor(1.)
king vs apple: tensor(0.)


## Cosine Similarity (Better Comparison)

In [16]:
def cosine_similarity(v1, v2):
    return torch.dot(v1, v2) / (torch.norm(v1) * torch.norm(v2))

print("king vs queen = ", cosine_similarity(manual_embeddings['king'], manual_embeddings['queen']))
print("king vs man = ", cosine_similarity(manual_embeddings['king'], manual_embeddings['man']))

king vs queen =  tensor(0.3333)
king vs man =  tensor(0.8165)


## Using BERT

##  What is "Transformers"?

- It is a massive library created by a company called **Hugging Face.**
- A **"Library of Brains"**: It gives you instant access to thousands of pre-trained AI models like `BERT`, `GPT-2`, and `Llama`.
- **The Bridge:** It handles all the difficult math (tensors, neural networks) using PyTorch so you can focus on building your app

- The library is built around three main classes that work together:
- 1. **Tokenizer:** Converts your text into numbers.
- 2. **Model:** The "brain" that processes those numbers and understands the context.
- 3. **Config:** The settings that tell the model how many `"layers"` or `"neurons"` to use

In [25]:
import torch
from transformers import BertTokenizer, BertModel

In [ ]:
# Load tokenizer 
# Loading the Brain (The Weights)
# This cell downloads the "frozen knowledge" of BERT from the internet (the first time you run it).
# The Tokenizer loads the specific 30,522-word dictionary BERT was trained on.
# The Model loads the 110 million "weights" (synapses) that allow it to understand grammar and context.
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased") # Loads pretrained BERT weights (12-layer Transformer encoder).

# inference mode
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2762.30it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
text = "King and queen rule the kingdom"

inputs = tokenizer(
    text,
    return_tensors="pt", # "pt" tells it to return PyTorch Tensors
    padding=True,
    truncation=True,
)

print(inputs)

{'input_ids': tensor([[ 101, 2332, 1998, 3035, 3627, 1996, 2983,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
#  The Inference Block (The "Thinking" Phase)
# "We are only using the model, not training it." 
# It turns off the math used for learning, which saves a lot of memory and makes the code run much faster.
# This feeds your tokenized text into the 12 layers of BERT. 
# The ** simply unpacks the dictionary (IDs, attention masks, etc.) so the model can read them.
# last_hidden_state :This is the "Final Answer" from the last layer of BERT. 
# It is a giant 3D tensor (tensor of tensors) containing a 768-dimensional vector for every single word in your sentence.
with torch.no_grad(): # 5. Disable gradient computation so it Saves memory, Speeds up inference
    outputs = model(**inputs) # Forward pass through BERT, Feeds tokenized input into BERT
    # nternally:
    # 1. Embedding layer (token + positional + segment embeddings)
    # 2. 12 Transformer encoder layers
    # 3. Self-attention computes contextual relationships 

# Extract hidden state
# Shape: (batch_size, sequence_length, hidden_size)
# For BERT base → hidden_size = 768
# Each token now has a context-aware vector
last_hidden_state = outputs.last_hidden_state   
last_hidden_state  # meaning-rich vectors for each token

tensor([[[-0.1312,  0.4706,  0.2847,  ..., -0.6902, -0.0541,  0.4395],
         [-0.1737, -0.0729,  0.5164,  ..., -0.9206, -0.2268, -0.6300],
         [-0.2392,  0.5482,  0.0452,  ..., -0.5399, -0.1975,  0.0848],
         ...,
         [-0.2447,  0.0071,  0.3242,  ..., -0.4087, -0.2076, -0.3365],
         [ 0.1948, -0.3012,  0.3505,  ..., -0.5209, -0.2106, -0.0634],
         [ 0.8555,  0.2338, -0.3292,  ..., -0.0679, -0.4858, -0.1758]]])